In [1]:
import wrds
import pandas as pd
from pathlib import Path

In [2]:
wrds_db = wrds.Connection()

Loading library list...
Done


In [ ]:

base = Path.cwd()
data_dir = base / "DATA"
raw_dir = data_dir / "raw"

raw_dir.mkdir(parents=True, exist_ok=True)





In [4]:
paper_countries = [
    "USA","JPN","CHN","IND","KOR","HKG","TWN","FRA","GBR","THA",
    "AUS","SGP","SWE","ZAF","POL","ISR","VNM","ITA","TUR","CHE",
    "IDN","GRC","PHL","NOR","LKA","DNK","FIN","SAU","JOR","EGY",
    "ESP","KWT"
]

In [ ]:
id_cols = [
    "id",
    "eom",
    "excntry",
    "gvkey",
    "permno",
    "size_grp",
    "me"
]



NameError: name 'feature_cols' is not defined

In [ ]:
paper36_candidates = [
    "be_me",              # bm
    "at_gr1",             # agr
    "ret_1_0",            # mom_1
    "ret_6_1",            # mom_6
    "ret_12_1",           # mom_12
    "rmax1_21d",          # maxret
    "rvol_21d",           # retvol
    "ni_be",              # roe
    "ni_me",              # ep
    "sale_gr1",           # sgr
    "sale_me",            # sp
    "div12m_me",          # dy
    "ami_126d",           # ill
    "dolvol_126d",        # dolvol
    "dolvol_var_126d",    # stddolvol
    "turnover_126d",      # turn
    "turnover_var_126d",  # stdturn
    "be_gr1a",            # egr
    "debt_me",            # lev proxy
    "at_me",              # asset/market related predictor
    "resff3_6_1",         # residual momentum proxy
    "resff3_12_1",        # residual momentum proxy
    "niq_be",             # alternate roe-style variable
    "ocf_me",             # cfp-style proxy if available
    "fcf_me",             # cfp-style proxy if available
    "oaccruals_at",       # acc proxy if available
    "taccruals_at",       # acc / absacc proxy if available
    "oaccruals_ni",       # pctacc proxy
    "taccruals_ni",       # pctacc proxy
    "cop_at",             # cash profitability style proxy if available
    "ope_be",             # profitability style proxy if available
    "eqnpo_me",           # equity issuance / related
    "ebit_sale",          # profit margin style base
    "sale_be",            # sales/book type
    "debt_gr3",           # debt growth proxy
    "tax_gr1a"            # accounting growth style extra predictor
]


In [ ]:
feature_cols = [col for col in paper36_candidates if col in available_cols]
missing_cols = [col for col in paper36_candidates if col not in available_cols]

print("Requested candidate features:", len(paper36_candidates))
print("Available feature columns:", len(feature_cols))
print("Missing candidate columns:", len(missing_cols))

In [ ]:
all_cols = id_cols + feature_cols
select_clause = ", ".join(all_cols)

print("Total columns to download:", len(all_cols))
print(select_clause)

Total columns to download: 43
id, eom, excntry, gvkey, permno, size_grp, me, be_me, at_gr1, ret_1_0, ret_6_1, ret_12_1, rmax1_21d, rvol_21d, ni_be, ni_me, sale_gr1, sale_me, div12m_me, ami_126d, dolvol_126d, dolvol_var_126d, turnover_126d, turnover_var_126d, be_gr1a, debt_me, at_me, resff3_6_1, resff3_12_1, niq_be, ocf_me, fcf_me, oaccruals_at, taccruals_at, oaccruals_ni, taccruals_ni, cop_at, ope_be, eqnpo_me, ebit_sale, sale_be, debt_gr3, tax_gr1a


In [ ]:
n_countries = len(paper_countries)
total_rows = 0
downloaded = []
skipped = []
failed = []

for i, country in enumerate(paper_countries, start=1):
    pct = 100 * i / n_countries
    country_dir = raw_dir / country
    country_dir.mkdir(parents=True, exist_ok=True)

    file_path = country_dir / f"{country}.parquet"

    print("\n" + "=" * 70)
    print(f"[{i}/{n_countries}] {pct:.1f}% complete | {country}")

    if file_path.exists():
        print(f"Skipped {country}: file already exists at {file_path}")
        skipped.append(country)
        continue

    sql_query = f"""
    SELECT {select_clause}
    FROM contrib.global_factor
    WHERE common = 1
      AND exch_main = 1
      AND primary_sec = 1
      AND obs_main = 1
      AND excntry = '{country}'
    """

    try:
        df_country = wrds_db.raw_sql(sql_query)

        n_rows = len(df_country)
        total_rows += n_rows

        print(f"Rows downloaded for {country}: {n_rows:,}")
        print(f"Cumulative rows downloaded: {total_rows:,}")

        df_country.to_parquet(file_path, index=False)

        print(f"Saved: {file_path}")
        downloaded.append(country)

        # optional quick peek
        print(df_country.head(2))

        # free memory explicitly
        del df_country

    except Exception as e:
        print(f"Error for {country}: {e}")
        failed.append(country)


[1/32] 3.1% complete | USA
Rows downloaded for USA: 4,262,872
Cumulative rows downloaded: 4,262,872
Saved: /Users/joshaijer/Documents/factor_model_project/DATA/raw/USA/USA.parquet
            id         eom excntry   gvkey  permno size_grp    me  be_me  \
0  101127101.0  1962-02-28     USA  011271    <NA>     <NA>  <NA>   <NA>   
1  100112401.0  1962-01-31     USA  001124    <NA>     <NA>  <NA>   <NA>   

     at_gr1  ret_1_0  ...  taccruals_at  oaccruals_ni  taccruals_ni    cop_at  \
0  0.075758     <NA>  ...      0.055493      0.691228      0.691228  0.159577   
1      <NA>     <NA>  ...          <NA>          <NA>          <NA>      <NA>   

     ope_be  eqnpo_me  ebit_sale   sale_be  debt_gr3  tax_gr1a  
0  0.263276      <NA>   0.099616  2.063103      <NA> -0.002817  
1      <NA>      <NA>       <NA>      <NA>      <NA>      <NA>  

[2 rows x 43 columns]

[2/32] 6.2% complete | JPN


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/wrds/sql.py:591: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  full_df = pd.concat([full_df, chunk])


Rows downloaded for JPN: 1,501,066
Cumulative rows downloaded: 5,763,938
Saved: /Users/joshaijer/Documents/factor_model_project/DATA/raw/JPN/JPN.parquet
            id         eom excntry   gvkey permno size_grp        me  be_me  \
0  320912101.0  1986-12-31     JPN  209121   <NA>    micro  99.81451   <NA>   
1  320912201.0  1986-12-31     JPN  209122   <NA>    micro  19.84468   <NA>   

   at_gr1  ret_1_0  ...  taccruals_at  oaccruals_ni  taccruals_ni  cop_at  \
0    <NA>  0.01191  ...          <NA>          <NA>          <NA>    <NA>   
1    <NA>     <NA>  ...          <NA>          <NA>          <NA>    <NA>   

   ope_be  eqnpo_me  ebit_sale  sale_be  debt_gr3  tax_gr1a  
0    <NA>      <NA>       <NA>     <NA>      <NA>      <NA>  
1    <NA>      <NA>       <NA>     <NA>      <NA>      <NA>  

[2 rows x 43 columns]

[3/32] 9.4% complete | CHN
Rows downloaded for CHN: 747,691
Cumulative rows downloaded: 6,511,629
Saved: /Users/joshaijer/Documents/factor_model_project/DATA/raw/CHN/C

In [ ]:
print("\n" + "=" * 70)
print("DOWNLOAD SUMMARY")
print("=" * 70)
print(f"Downloaded countries ({len(downloaded)}): {downloaded}")
print(f"Skipped countries ({len(skipped)}): {skipped}")
print(f"Failed countries ({len(failed)}): {failed}")
print(f"Total rows downloaded this run: {total_rows:,}")


DOWNLOAD SUMMARY
Downloaded countries (32): ['USA', 'JPN', 'CHN', 'IND', 'KOR', 'HKG', 'TWN', 'FRA', 'GBR', 'THA', 'AUS', 'SGP', 'SWE', 'ZAF', 'POL', 'ISR', 'VNM', 'ITA', 'TUR', 'CHE', 'IDN', 'GRC', 'PHL', 'NOR', 'LKA', 'DNK', 'FIN', 'SAU', 'JOR', 'EGY', 'ESP', 'KWT']
Skipped countries (0): []
Failed countries (0): []
Total rows downloaded this run: 12,737,911
